In [16]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import seaborn as sns
import zipfile

## Canonicalization

In [17]:
county_and_state = pd.read_csv('county_and_state.csv')
    
county_and_pop = pd.read_csv('county_and_population.csv')    

Suppose we'd like to join these two tables. Unfortunately, we can't, because the strings representing the county names don't match, as seen below.

In [18]:
county_and_state

,County,State
0,De Witt County,IL
1,Lac qui Parle County,MN
2,Lewis and Clark County,MT
3,St John the Baptist Parish,LS


In [19]:
county_and_pop

,County,Population
0,DeWitt,16798
1,Lac Qui Parle,8067
2,Lewis & Clark,55716
3,St. John the Baptist,43044


 Before we can join them, we'll do what we call **canonicalization**.

Canonicalization: A process for converting data that has more than one possible representation into a "standard", "normal", or canonical form (definition via Wikipedia).

In [20]:
county_and_state["County"][0]

'De Witt County'

In [21]:
county_and_state["County"][0].lower()

'de witt county'

In [22]:
county_and_state["County"][0].lower().replace(' ', '')

'dewittcounty'

In [23]:
county_and_state["County"][0].lower().replace(' ', '').replace('county', '')

'dewitt'

Now do this to every county in the data.

In [30]:
county_and_pop["clean_county"] = (county_and_state["County"]
        .str.lower()               # lower case
        .str.replace(' ', '')      # remove spaces
        .str.replace('&', 'and')   # replace &
        .str.replace('.', '')      # remove dot
        .str.replace('county', '') # remove county
        .str.replace('parish', '') # remove parish
    )

In [31]:
county_and_state["clean_county"] = (county_and_state["County"]
        .str.lower()               # lower case
        .str.replace(' ', '')      # remove spaces
        .str.replace('&', 'and')   # replace &
        .str.replace('.', '')      # remove dot
        .str.replace('county', '') # remove county
        .str.replace('parish', '') # remove parish
    )

In [32]:
county_and_state.head()

,County,State,clean_county
0,De Witt County,IL,dewitt
1,Lac qui Parle County,MN,lacquiparle
2,Lewis and Clark County,MT,lewisandclark
3,St John the Baptist Parish,LS,stjohnthebaptist


In [33]:
county_and_pop

,County,Population,clean_county
0,DeWitt,16798,dewitt
1,Lac Qui Parle,8067,lacquiparle
2,Lewis & Clark,55716,lewisandclark
3,St. John the Baptist,43044,stjohnthebaptist


In [34]:
county_and_pop.merge(county_and_state,
                     left_on = 'clean_county', right_on = 'clean_county')

,County_x,Population,clean_county,County_y,State
0,DeWitt,16798,dewitt,De Witt County,IL
1,Lac Qui Parle,8067,lacquiparle,Lac qui Parle County,MN
2,Lewis & Clark,55716,lewisandclark,Lewis and Clark County,MT
3,St. John the Baptist,43044,stjohnthebaptist,St John the Baptist Parish,LS


## Processing Data from a Text Log Using Basic Python

In [39]:
with open('log.txt', 'r') as f:
    log_lines = f.readlines()

In [40]:
log_lines

['169.237.46.168 - - [26/Jan/2014:10:47:58 -0800] "GET /stat141/Winter04/ HTTP/1.1" 200 2585 "http://anson.ucdavis.edu/courses/"\n',
 '193.205.203.3 - - [2/Feb/2005:17:23:6 -0800] "GET /stat141/Notes/dim.html HTTP/1.0" 404 302 "http://eeyore.ucdavis.edu/stat141/Notes/session.html"\n',
 '169.237.46.240 - "" [3/Feb/2006:10:18:37 -0800] "GET /stat141/homework/Solutions/hw1Sol.pdf HTTP/1.1"\n']

Suppose we want to extract the day, month, year, hour, minutes, seconds, and timezone. Looking at the data, we see that these items are not in a fixed position relative to the beginning of the string. That is, slicing by some fixed offset isn't going to work.

In [41]:
log_lines[0][20:31]

'26/Jan/2014'

In [42]:
log_lines[1][20:31]

'/Feb/2005:1'

Instead, we'll need to use some more sophisticated thinking. Let's focus on only the first line of the file.

In [45]:
first = log_lines[0]
first

'169.237.46.168 - - [26/Jan/2014:10:47:58 -0800] "GET /stat141/Winter04/ HTTP/1.1" 200 2585 "http://anson.ucdavis.edu/courses/"\n'

In [46]:
first.split("[")

['169.237.46.168 - - ',
 '26/Jan/2014:10:47:58 -0800] "GET /stat141/Winter04/ HTTP/1.1" 200 2585 "http://anson.ucdavis.edu/courses/"\n']

In [47]:
pertinent = first.split("[")[1].split(']')[0]
pertinent

'26/Jan/2014:10:47:58 -0800'

In [48]:
pertinent = first.split("[")[1].split(']')[0]
day, month, rest = pertinent.split('/')
year, hour, minute, rest = rest.split(':')
seconds, time_zone = rest.split(' ')
day, month, year, hour, minute, seconds, time_zone

('26', 'Jan', '2014', '10', '47', '58', '-0800')

A much more sophisticated but common approach is to extract the information we need using a regular expression. 

In [50]:
import re
pattern = r'\[(\d+)/(\w+)/(\d+):(\d+):(\d+):(\d+) (.+)\]'
day, month, year, hour, minute, second, time_zone = re.search(pattern, first).groups()
year, month, day, hour, minute, second, time_zone

('2014', 'Jan', '26', '10', '47', '58', '-0800')

In [51]:
re.search(pattern, first)[0]

'[26/Jan/2014:10:47:58 -0800]'

Or alternately using the `findall` method:

In [52]:
import re
pattern = r'\[(\d+)/(\w+)/(\d+):(\d+):(\d+):(\d+) (.+)\]'
day, month, year, hour, minute, second, time_zone = re.findall(pattern, first)[0]
year, month, day, hour, minute, second, time_zone

('2014', 'Jan', '26', '10', '47', '58', '-0800')

Note: We can return the results as a Series:

In [53]:
cols = ["Day", "Month", "Year", "Hour", "Minute", "Second", "Time Zone"]
def log_entry_to_series(line):
    return pd.Series(re.search(pattern, line).groups(), index = cols)

log_entry_to_series(first)

Day             26
Month          Jan
Year          2014
Hour            10
Minute          47
Second          58
Time Zone    -0800
dtype: object

And using this function we can create a DataFrame of all the time information.

In [54]:
log_info = pd.DataFrame(columns=cols, index=range(3))

for i in np.arange(0,len(log_lines)):
    log_info.loc[i] = log_entry_to_series(log_lines[i])

log_info

,Day,Month,Year,Hour,Minute,Second,Time Zone
0,26,Jan,2014,10,47,58,-0800
1,2,Feb,2005,17,23,6,-0800
2,3,Feb,2006,10,18,37,-0800


## Restaurant Data

In this example, we will show how regexes can allow us to track quantitative data across categories defined by the appearance of various text fields.

In this example we'll see how the presence of certain keywords can affect quantitative data, e.g. how do restaurant health scores vary as a function of the number of violations that mention "vermin"?

In [55]:
vio = pd.read_csv('violations.csv', header=0, names=['id', 'date', 'desc'])
desc = vio['desc']
vio.head()

,id,date,desc
0,19,20171211,Inadequate food safety knowledge or lack of ce...
1,19,20171211,Unapproved or unmaintained equipment or utensils
2,19,20160513,Unapproved or unmaintained equipment or utensi...
3,19,20160513,Unclean or degraded floors walls or ceilings ...
4,19,20160513,Food safety certificate or food handler card n...


In [56]:
counts = desc.value_counts()

counts[:10]

desc
Unclean or degraded floors walls or ceilings                          999
Unapproved or unmaintained equipment or utensils                      659
Inadequately cleaned or sanitized food contact surfaces               493
Improper food storage                                                 476
Inadequate and inaccessible handwashing facilities                    467
Moderate risk food holding temperature                                452
Wiping cloths not clean or properly stored or inadequate sanitizer    418
Moderate risk vermin infestation                                      374
Unclean nonfood contact surfaces                                      369
Food safety certificate or food handler card not available            353
Name: count, dtype: int64

In [57]:
# Hmmm...
counts[50:60]

desc
Unclean or degraded floors walls or ceilings  [ date violation corrected: 11/29/2017 ]              16
Unclean or degraded floors walls or ceilings  [ date violation corrected: 9/19/2017 ]               16
Inadequate HACCP plan record keeping                                                                16
Unclean or degraded floors walls or ceilings  [ date violation corrected: 11/27/2017 ]              15
Unclean or degraded floors walls or ceilings  [ date violation corrected: 12/7/2017 ]               15
Inadequately cleaned or sanitized food contact surfaces  [ date violation corrected: 9/26/2017 ]    14
Unclean or degraded floors walls or ceilings  [ date violation corrected: 11/28/2017 ]              14
Unclean or degraded floors walls or ceilings  [ date violation corrected: 9/6/2017 ]                14
Unapproved or unmaintained equipment or utensils  [ date violation corrected: 9/19/2017 ]           14
Unapproved  living quarters in food facility                        

In [58]:
#Use regular expressions to cut out the extra info in square braces.
vio['clean_desc'] = (vio['desc']
             .str.replace(r"\s*\[.*\]$", '')
             .str.strip()
             .str.lower())
vio.head()

,id,date,desc,clean_desc
0,19,20171211,Inadequate food safety knowledge or lack of ce...,inadequate food safety knowledge or lack of ce...
1,19,20171211,Unapproved or unmaintained equipment or utensils,unapproved or unmaintained equipment or utensils
2,19,20160513,Unapproved or unmaintained equipment or utensi...,unapproved or unmaintained equipment or utensi...
3,19,20160513,Unclean or degraded floors walls or ceilings ...,unclean or degraded floors walls or ceilings ...
4,19,20160513,Food safety certificate or food handler card n...,food safety certificate or food handler card n...


In [59]:
help(pd.Series.str.strip)

Help on function strip in module pandas.core.strings.accessor:

strip(self, to_strip=None)
    Remove leading and trailing characters.

    Strip whitespaces (including newlines) or a set of specified characters
    from each string in the Series/Index from left and right sides.
    Replaces any non-strings in Series with NaNs.
    Equivalent to :meth:`str.strip`.

    Parameters
    ----------
    to_strip : str or None, default None
        Specifying the set of characters to be removed.
        All combinations of this set of characters will be stripped.
        If None then whitespaces are removed.

    Returns
    -------
    Series or Index of object

    See Also
    --------
    Series.str.strip : Remove leading and trailing characters in Series/Index.
    Series.str.lstrip : Remove leading characters in Series/Index.
    Series.str.rstrip : Remove trailing characters in Series/Index.

    Examples
    --------
    >>> s = pd.Series(['1. Ant.  ', '2. Bee!\n', '3. Cat?\t', np.na

In [60]:
vio['clean_desc'].value_counts().head() 

clean_desc
unclean or degraded floors walls or ceilings               999
unapproved or unmaintained equipment or utensils           659
inadequately cleaned or sanitized food contact surfaces    493
improper food storage                                      476
inadequate and inaccessible handwashing facilities         467
Name: count, dtype: int64

In [61]:
#use regular expressions to assign new features for the presence of various keywords
with_features = pd.DataFrame(vio)

In [62]:
with_features["is_clean"] = vio['clean_desc'].str.contains('clean|sanit')
with_features["is_high_risk"] = vio['clean_desc'].str.contains('high risk')
with_features["is_vermin"] = vio['clean_desc'].str.contains('vermin')
with_features["is_surface"] = vio['clean_desc'].str.contains('wall|ceiling|floor|surface')
with_features["is_human"] = vio['clean_desc'].str.contains('hand|glove|hair|nail')
with_features["is_permit"] = vio['clean_desc'].str.contains('permit|certif')

with_features.head()

,id,date,desc,clean_desc,is_clean,is_high_risk,is_vermin,is_surface,is_human,is_permit
0,19,20171211,Inadequate food safety knowledge or lack of ce...,inadequate food safety knowledge or lack of ce...,False,False,False,False,False,True
1,19,20171211,Unapproved or unmaintained equipment or utensils,unapproved or unmaintained equipment or utensils,False,False,False,False,False,False
2,19,20160513,Unapproved or unmaintained equipment or utensi...,unapproved or unmaintained equipment or utensi...,False,False,False,False,False,False
3,19,20160513,Unclean or degraded floors walls or ceilings ...,unclean or degraded floors walls or ceilings ...,True,False,False,True,False,False
4,19,20160513,Food safety certificate or food handler card n...,food safety certificate or food handler card n...,False,False,False,False,True,True


In [63]:
count_features = with_features.groupby(['id', 'date']).sum().reset_index()
count_features.iloc[255:260, :]

,id,date,desc,clean_desc,is_clean,is_high_risk,is_vermin,is_surface,is_human,is_permit
255,489,20150728,Unclean or degraded floors walls or ceilings ...,unclean or degraded floors walls or ceilings ...,5,0,2,3,0,0
256,489,20150807,Unapproved or unmaintained equipment or utensi...,unapproved or unmaintained equipment or utensi...,1,0,0,1,0,0
257,489,20160308,High risk food holding temperature [ date vi...,high risk food holding temperature [ date vi...,2,2,1,0,1,0
258,489,20160721,Low risk vermin infestation [ date violation ...,low risk vermin infestation [ date violation ...,2,1,1,1,0,1
259,489,20161220,Inadequately cleaned or sanitized food contact...,inadequately cleaned or sanitized food contact...,3,0,1,2,0,0


Now tidy up the data

In [64]:
#use melt to turn our columns into a single variable (columns)
#the granularity of the resulting frame is a violation type in a given inspection
violations_long = pd.melt(count_features.drop(["desc","clean_desc"], axis = 1), id_vars=['id', 'date'],
            var_name='violation', value_name='num_vios')
violations_long.sort_values(["id", "date"]).head(13)

,id,date,violation,num_vios
0,19,20160513,is_clean,1
12262,19,20160513,is_high_risk,0
24524,19,20160513,is_vermin,0
36786,19,20160513,is_surface,1
49048,19,20160513,is_human,1
61310,19,20160513,is_permit,1
1,19,20171211,is_clean,0
12263,19,20171211,is_high_risk,0
24525,19,20171211,is_vermin,0
36787,19,20171211,is_surface,0


In [65]:
#read in the scores
ins = pd.read_csv('inspections.csv', header=0, usecols=[0, 1, 2], names=['id', 'score', 'date'])
ins.head()

,id,score,date
0,19,94,20160513
1,19,94,20171211
2,24,98,20171101
3,24,98,20161005
4,24,96,20160311


In [66]:
#join scores with the table broken down by violation type
violation_type_and_scores = violations_long.merge(ins, left_on=['id', 'date'], right_on=['id', 'date'])
violation_type_and_scores.head(12)

,id,date,violation,num_vios,score
0,19,20160513,is_clean,1,94
1,19,20171211,is_clean,0,94
2,24,20160311,is_clean,2,96
3,24,20161005,is_clean,1,98
4,24,20171101,is_clean,0,98
5,31,20151204,is_clean,0,98
6,45,20160104,is_clean,3,78
7,45,20160614,is_clean,1,84
8,45,20170307,is_clean,3,88
9,45,20170914,is_clean,2,85
